In [47]:
# System
import os
import shutil
import io
import datetime
import zipfile
import requests
import subprocess
import glob
import requests
import bs4
import time

# Data Analysis
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark import SparkConf,SparkContext

pd.set_option('display.max_columns', None)

In [48]:
class Init:
    """
    """
    def __init__(self):
        cwd = os.getcwd()
        
        self.preprocess = os.path.join(cwd,'preproc')
        self.retro = os.path.join(self.preprocess,'retro')
        self.bbref = os.path.join(self.preprocess,'bbref')
        self.lahman = os.path.join(self.preprocess,'lahman')
        self.inputs = os.path.join(cwd,'inputs')
        
        for i in [self.preprocess,self.retro,self.bbref,self.lahman,self.inputs]:
            if os.path.exists(i):
                shutil.rmtree(i)
            os.makedirs(i)

In [20]:
Init()

In [59]:
class RetroSheet:
    """
    """
    def __init__(self):
        # file setup
        cwd = os.getcwd()
        self.retro = os.path.join(cwd,'preproc','retro')
        self.events_dir = os.path.join(self.retro,'events')
        self.games_dir = os.path.join(self.retro,'games')
        self.exe_dir = os.path.join(self.retro,'exe')
        self.inputs = os.path.join(cwd,'inputs')
        # retrosheet urls for data downloads
        self.base_url = 'https://www.retrosheet.org'
        self.events_url = self.base_url+'/events/{}eve.zip' # play by play events
        self.games_url = self.base_url+'/gamelogs/gl{}.zip' # game logs
        # data extract execution files
        self.bevent_exe_url = self.base_url+'/bevent.zip'
        self.game_exe_url = self.base_url+'/bgame.zip'
        self.box_exe_url = self.base_url+'/box.zip'        
        # path for extracted data
        self.events_fpath = os.path.join(self.retro,'events.txt')
        self.games_fpath = os.path.join(self.retro,'games.txt')
        
        # self.event_columns_fpath = os.path.join(self.retro,'event_columns.txt')
        # self.game_columns_fpath = os.path.join(self.retro,'game_columns.txt')
        
        self.event_columns = None
        self.game_columns = None
        self._extract_columns()
        
        # self.events = None
        # self.games = None
        
        # self.event_mapper = None
        # self._mapper()
        
        # self.rosters = None
        # self._rosters()
        
        # self.teams = None
        # self._teams()
        
        # self.events_csv = os.path.join(self.retro,'events.csv')
        # self.games_csv = os.path.join(self.retro,'games.csv')
    
    def _extract_columns(self):
        self.event_columns = {
            '0': 'game_id','1': 'vis_team','2': 'inning',
            '3': 'batting_team','4': 'outs','5': 'balls',
            '6': 'strikes','7': 'pitch_sequence','8': 'vis_score',
            '9': 'home_score','10': 'batter','11': 'batter_hand',
            '14': 'pitcher','15': 'pitcher_hand','18':'catcher',
            '19':'first_base','20':'second_base','21':'third_base',
            '22':'short_stop','23':'left_field','24':'center_field',
            '25':'right_field','29': 'event_text','33':'lineup_position',
            '34':'event_type','40':'outs_on_play','43':'rbi',
            '50':'hit_location','96': 'event_num',
        }
        
#         game_cols = {}
#         # copy and paste these from data use webpage into file
#         with open(self.game_columns_fpath,'r') as f:
#             content = f.read()
#             for line in content.split('\n')[2:]:
#                 k,v = line.replace('*','').split(' ',1)
#                 game_cols[k.strip()] = v.strip()
#         self.game_columns = game_cols
    
    # def _mapper(self):
    #     self.event_mapper = {
    #         'event_type': {
    #             0: 'Unknown event',1: 'No event',2: 'Generic out',
    #             3: 'Strikeout',4: 'Stolen base',5: 'Defensive indifference',
    #             6: 'Caught stealing',7: 'Pickoff error',8: 'Pickoff',
    #             9: 'Wild pitch',10: 'Passed ball',11: 'Balk',
    #             12: 'Other advance',13: 'Foul error',14: 'Walk',
    #             15: 'Intentional walk',16: 'Hit by pitch',17: 'Interference',
    #             18: 'Error',19: 'Fielders choice',20: 'Single',
    #             21: 'Double',22: 'Triple',23: 'Home run',
    #             24: 'Missing play',
    #         }
    #     }
    
    # def _teams(self):
    #     teams = {}
    #     team_fields = ['team_id','league','city','name']
    #     team_fpaths = glob.glob(os.path.join(self.retro,'TEAM*'))
    #     for fpath in team_fpaths:
    #         filename = os.path.basename(fpath)
    #         year = filename[-4:]
    #         team_info = pd.read_csv(fpath,header=None,names=team_fields).set_index('team_id')#.to_dict(orient='index')
    #         teams[year] = team_info
    #     self.teams = teams
    
    # def _rosters(self):
    #     rosters = {}
    #     roster_fields = [
    #         'player_id','last_name','first_name',
    #         'throwing_hand','batting_hand','team_id',
    #         'position'
    #     ]
    #     roster_fpaths = glob.glob(os.path.join(self.retro,'*.ROS'))
    #     for fpath in roster_fpaths:
    #         filename = os.path.basename(fpath)
    #         team_id = filename[:3]
    #         year = filename[3:7]
    #         roster = pd.read_csv(fpath,header=None,names=roster_fields).set_index('player_id')#.to_dict(orient='index')
    #         if rosters.get(year,None) is None:
    #             rosters[year] = {}
    #         rosters[year][team_id] = roster
    #     self.rosters = rosters
        
    def download(self,years):
        for url in [self.bevent_url,self.bgame_url]:
            time.sleep(1)
            
            r = requests.get(url)
            z = zipfile.ZipFile(io.BytesIO(r.content))
            z.extractall(self.exe_dir)
        
        self.years = years
        for year in years:
            for url,loc in {self.events_url:self.events_dir,self.games_url:self.games_dir}.items():
                time.sleep(1)
                
                u = url.format(year)
                r = requests.get(u)
                z = zipfile.ZipFile(io.BytesIO(r.content))
                z.extractall(loc)
        
        # then move team files out into same folder as BEVENT and rename to TEAM
    
    def extract(self):
        events = '{FOLDER}/BEVENT.EXE -f {COLUMNS} {BASE}/*******.EV* > {EVENTS}'.format(
            FOLDER = self.exe_dir,
            COLUMNS = ','.join([i for i in self.event_columns.keys()]),
            BASE = self.events_dir,
            EVENTS = self.events_fpath
        )
        games = '{FOLDER}/BGAME.EXE {BASE}/*******.TXT* > {GAMES}'.format(
            FOLDER = self.retro,
            BASE = self.games_dir,
            GAMES = self.games_fpath
        )
        
        print(events)
        print(games)
        
        subprocess.check_call(events.split(' '))
        subprocess.check_call(games.split(' '))
    
    # def load(self):
    #     self.events = pd.read_csv(self.events_fpath,encoding='utf-16',header=None,names=[i for i in self.event_columns.values()])
    #     self.games = pd.read_csv(self.games_fpath,encoding='utf-16',header=None,names=[i for i in self.game_columns.values()])
    
#     def process(self):
#         self.events['home_team'] = self.events.game_id.str[:3]
#         self.events['game_date'] = pd.to_datetime(self.events.game_id.str[3:-1],format='%Y%m%d')
#         for field,mapper in self.event_mapper.items():
#             self.events[field+'_desc'] = self.events[field].replace(mapper)
        
#         outcomes = [
#             'Strikeout','Generic out',
#             'Walk','Intentional walk',
#             'Single','Double','Triple','Homerun',
#             'Passed ball','Hit by pitch',
#             'Fielders choice','Other advance',
#         ]
#         self.events = self.events.loc[self.events.event_type_desc.isin(outcomes)]
        
    # def write(self):
    #     self.events.to_csv(self.events_csv,header=True,index=False)
    #     self.games.to_csv(self.games_csv,header=True,index=False)

In [60]:
retro = RetroSheet()
# retro.download(years=[2020,2021])

In [61]:
retro.extract()

/Users/tjordan/Documents/github/baseball/preproc/retro/exe/BEVENT.EXE -f 0,1,2,3,4,5,6,7,8,9,10,11,14,15,18,19,20,21,22,23,24,25,29,33,34,40,43,50,96 /Users/tjordan/Documents/github/baseball/preproc/retro/events/*******.EV* > /Users/tjordan/Documents/github/baseball/preproc/retro/events.txt
/Users/tjordan/Documents/github/baseball/preproc/retro/BGAME.EXE /Users/tjordan/Documents/github/baseball/preproc/retro/games/*******.TXT* > /Users/tjordan/Documents/github/baseball/preproc/retro/games.txt


PermissionError: [Errno 13] Permission denied: '/Users/tjordan/Documents/github/baseball/preproc/retro/exe/BEVENT.EXE'

In [53]:
retro.load()
retro.process()
retro.write()

In [61]:
class BballRef:
    """
    """
    def __init__(self):
        self.team_url = 'https://www.baseball-reference.com/teams/{TEAM_ID}/2022.shtml'
        self.team_performance_url = 'https://www.baseball-reference.com/leagues/majors/2022.shtml'
        self.team_standings_url = 'https://www.baseball-reference.com/leagues/MLB-standings.shtml'
        
        self.teams = [
            'ARI', 'ATL', 'BAL', 'BOS',
            'CHW', 'CHC', 'CIN', 'CLE',
            'COL', 'DET', 'HOU', 'KCR',
            'LAA', 'LAD', 'MIA', 'MIL',
            'MIN', 'NYY', 'NYM', 'OAK',
            'PHI', 'PIT', 'SDP', 'SEA',
            'SFG', 'STL', 'TBR', 'TEX',
            'TOR', 'WSN',
        ]
        
        self.batting = None
        self.pitching = None
        self.standings = None
        self.team_batting = None
        self.team_fielding = None
        self.team_pitching = None
        
        cwd = os.getcwd()
        self.batting_csv = os.path.join(cwd,'bbref','batting.csv')
        self.pitching_csv = os.path.join(cwd,'bbref','pitching.csv')
        self.standings = os.path.join(cwd,'bbref','standings.csv')
        self.team_batting_csv = os.path.join(cwd,'bbref','team_batting.csv')
        self.team_fielding_csv = os.path.join(cwd,'bbref','team_fielding.csv')
        self.team_pitching_csv = os.path.join(cwd,'bbref','team_pitching.csv')
    
    def download(self):
        batting_teams = []
        pitching_teams = []
        
        for team in self.teams:
            time.sleep(1)
            
            url = self.team_url.format(TEAM_ID=team)
            r = requests.get(url)
            soup = bs4.BeautifulSoup(r.text,'html.parser')
            
            for table_id in ['team_batting','team_pitching']:
                table = soup.find(name='table',attrs={'id':table_id})
                header = [i.text for i in table.find(name='thead').find_all(name='th',attrs={'scope':'col'})]
                rows = table.find(name='tbody').find_all(name='tr')

                data = []
                for row in rows:
                    attr = [i.text for i in row.find_all(name='td')]
                    idx = row.find(name='th').text
                    data.append([team]+[idx]+attr)

                df = pd.DataFrame(data,columns=['Team']+header)
                
                if table_id == 'team_batting':
                    batting_teams.append(df)
                elif table_id == 'team_pitching':
                    pitching_teams.append(df)
        
        self.batting = pd.concat(batting_teams)
        self.pitching = pd.concat(pitching_teams)
        
        time.sleep(1)
        r = requests.get(self.team_standings_url)
        soup = bs4.BeautifulSoup(r.text,'html.parser')
        
#         for table_id in ['team_standings']:                                    
#             table = soup.find(attrs={'id':table_id})
#             header = [i.text for i in table.find(name='thead').find_all(name='th',attrs={'scope':'col'})]
#             rows = table.find(name='tbody').find_all(name='tr')
            
#             data = []
#             for row in rows:
#                 attr = [i.text for i in row.find_all(name='td')]
#                 idx = row.find(name='th').text
#                 data.append([idx]+attr)

#             df = pd.DataFrame(data,columns=header)

#             if table_id == 'team_standings':
#                 self.standings = df
        
        time.sleep(1)
        r = requests.get(self.team_performance_url)
        soup = bs4.BeautifulSoup(r.text,'html.parser')
        
        for table_id in ['teams_standard_batting']:#,'teams_standard_pitching','teams_standard_fielding']:          
            table = soup.find(attrs={'id':table_id})
            header = [i.text for i in table.find(name='thead').find_all(name='th',attrs={'scope':'col'})]
            rows = table.find(name='tbody').find_all(name='tr')
            
            data = []
            for row in rows:
                attr = [i.text for i in row.find_all(name='td')]
                idx = row.find(name='th').text
                data.append([idx]+attr)

            df = pd.DataFrame(data,columns=header)

            if table_id == 'teams_standard_batting':
                self.team_batting = df
            elif table_id == 'teams_standard_pitching':
                self.team_pitching = df
            elif table_id == 'teams_standard_fielding':
                self.team_fielding = df

    def process(self):
        None
    
    def write(self):
        self.batting.to_csv(self.batting_csv,index=False)
        self.pitching.to_csv(self.pitching_csv,index=False)
        self.standings.to_csv(self.standings_csv,index=False)
        self.team_batting.to_csv(self.team_batting_csv,index=False)
        self.team_fielding.to_csv(self.team_fielding_csv,index=False)
        self.team_pitching.to_csv(self.team_pitching_csv,index=False)

In [59]:
bbref = BballRef()
bbref.download()
bbref.process()
bbref.write()

AttributeError: 'NoneType' object has no attribute 'find'

In [8]:
class Lahman:
    """
    """
    def __init__(self):
        self.folder_path = os.path.join(os.getcwd(),'lahman')
        self.core = os.path.join(self.folder_path,'core')
        
        self.team_csv = os.path.join(self.folder_path,'team.csv')
        self.player_csv = os.path.join(self.folder_path,'player.csv')
        
        self.player_final = None
    
    def load(self):
        # self.allstar = pd.read_csv(os.path.join(self.core,'AllStarFull.csv'),header=0)
        # self.appearances = pd.read_csv(os.path.join(self.core,'Appearances.csv'),header=0)
        self.batting = pd.read_csv(os.path.join(self.core,'Batting.csv'),header=0)
        # self.batting_post = pd.read_csv(os.path.join(self.core,'BattingPost.csv'),header=0)
        self.fielding = pd.read_csv(os.path.join(self.core,'Fielding.csv'),header=0)
        # self.fielding_of = pd.read_csv(os.path.join(self.core,'FieldingOF.csv'),header=0)
        # self.field_of_split = pd.read_csv(os.path.join(self.core,'FieldingOFsplit.csv'),header=0)
        # self.fielding_post = pd.read_csv(os.path.join(self.core,'FieldingPost.csv'),header=0)
        # self.home_games = pd.read_csv(os.path.join(self.core,'HomeGames.csv'),header=0)
        # self.managers = pd.read_csv(os.path.join(self.core,'Managers.csv'),header=0)
        # self.managers_half = pd.read_csv(os.path.join(self.core,'ManagersHalf.csv'),header=0)
        # self.parks = pd.read_csv(os.path.join(self.core,'Parks.csv'),header=0)
        self.people = pd.read_csv(os.path.join(self.core,'People.csv'),header=0)
        self.pitching = pd.read_csv(os.path.join(self.core,'Pitching.csv'),header=0)
        # self.pitching_post = pd.read_csv(os.path.join(self.core,'PitchingPost.csv'),header=0)
        # self.series_post = pd.read_csv(os.path.join(self.core,'SeriesPost.csv'),header=0)
        self.teams = pd.read_csv(os.path.join(self.core,'Teams.csv'),header=0)
        # self.franchises = pd.read_csv(os.path.join(self.core,'TeamsFranchises.csv'),header=0)
        # self.teams_half = pd.read_csv(os.path.join(self.core,'TeamsHalf.csv'),header=0)
    
    def player(self):
        None
    
    def team(self):
        None
    
    def write(self):
        None

In [9]:
lahman = Lahman()
lahman.load()
# lahman.player()
# lahman.team()
lahman.write()

In [2]:
class Vegas:
    """
    """
    def __init__(self):
        cwd = os.getcwd()
        self.source = 'https://sports-statistics.com/sports-data/mlb-historical-odds-scores-datasets/'
        self.df_2018 = pd.read_excel(os.path.join(cwd,'vegas','mlb odds 2018.xlsx'))
        self.df_2019 = pd.read_excel(os.path.join(cwd,'vegas','mlb odds 2019.xlsx'))
        self.df_2020 = pd.read_excel(os.path.join(cwd,'vegas','mlb odds 2020.xlsx'))
        
        self.vegas_csv = os.path.join(cwd,'processed','vegas.csv')
    
    def process(self):
        None
    
    def write(self):
        None

In [75]:
class ETL:
    """
    """
    def __init__(self):
        cwd = os.getcwd()
        
        self.player = None
        self.team = None
        self.atbat = None
        
        self.player_csv = os.path.join(cwd,'processed','player.csv')
        self.team_csv = os.path.join(cwd,'processed','team.csv')
        self.atbat_csv = os.path.join(cwd,'processed','atbat.csv')
        
        self.files = {
            'bbref': {
                'batting': os.path.join(cwd,'bbref','batting.csv'),
                'pitching': os.path.join(cwd,'bbref','pitching.csv'),
                'standings': os.path.join(cwd,'bbref','standings.csv'),
                # 'team_batting': os.path.join(cwd,'bbref','team_batting.csv'),
                # 'team_fielding': os.path.join(cwd,'bbref','team_fielding.csv'),
                # 'team_pitching': os.path.join(cwd,'bbref','team_pitching.csv'),
            },
            'lahman': {
                'people': os.path.join(cwd,'lahman','core','People.csv'),
                'team': os.path.join(cwd,'lahman','core','Teams.csv'),
                'fielding': os.path.join(cwd,'lahman','core','Fielding.csv'),
                'pitching': os.path.join(cwd,'lahman','core','Pitching.csv'),
                'batting': os.path.join(cwd,'lahman','core','Batting.csv'),
            },
            'retrosheet': {
                'events': os.path.join(cwd,'retrosheet','events.csv'),
                'games': os.path.join(cwd,'retrosheet','games.csv'),
            },
            #'vegas': {},
        }
        self.spark = SparkSession \
                    .builder \
                    .appName("Baseball ETL") \
                    .getOrCreate()
    
    def load(self):
        for folder,files in self.files.items():
            for fname,fpath in files.items():
                df = self.spark.read.csv(fpath,header=True)
                name = '_'.join([folder,fname]).upper()
                df.createOrReplaceTempView(name)
    
    def player(self):
        sql = """
        """
        self.player = self.spark.sql(sql)
    
    def team(self):
        sql = """
        """
        self.team = self.spark.sql(sql)
    
    def atbat(self):
        sql = """
        """
        self.atbat = self.spark.sql(sql)

In [76]:
etl = ETL()

In [77]:
etl.load()

In [87]:
etl.spark.sql(
    '''
    select * from RETROSHEET_EVENTS limit 1
    '''
).show()

+------------+--------+------+------------+----+-----+-------+--------------+---------+----------+--------+-----------+--------+------------+--------+----------+-----------+----------+----------+----------+------------+-----------+----------+---------------+----------+------------+---+------------+---------+---------+----------+---------------+
|     game_id|vis_team|inning|batting_team|outs|balls|strikes|pitch_sequence|vis_score|home_score|  batter|batter_hand| pitcher|pitcher_hand| catcher|first_base|second_base|third_base|short_stop|left_field|center_field|right_field|event_text|lineup_position|event_type|outs_on_play|rbi|hit_location|event_num|home_team| game_date|event_type_desc|
+------------+--------+------+------------+----+-----+-------+--------------+---------+----------+--------+-----------+--------+------------+--------+----------+-----------+----------+----------+----------+------------+-----------+----------+---------------+----------+------------+---+------------+-------

In [84]:
etl.spark.sql(
    '''
    select * from LAHMAN_PEOPLE limit 1
    '''
).show()

+---------+---------+----------+--------+------------+----------+---------+---------+----------+--------+------------+----------+---------+---------+--------+-----------+------+------+----+------+----------+----------+--------+---------+
| playerID|birthYear|birthMonth|birthDay|birthCountry|birthState|birthCity|deathYear|deathMonth|deathDay|deathCountry|deathState|deathCity|nameFirst|nameLast|  nameGiven|weight|height|bats|throws|     debut| finalGame| retroID|  bbrefID|
+---------+---------+----------+--------+------------+----------+---------+---------+----------+--------+------------+----------+---------+---------+--------+-----------+------+------+----+------+----------+----------+--------+---------+
|aardsda01|     1981|        12|      27|         USA|        CO|   Denver|     null|      null|    null|        null|      null|     null|    David| Aardsma|David Allan|   215|    75|   R|     R|2004-04-06|2015-08-23|aardd001|aardsda01|
+---------+---------+----------+--------+-------

In [86]:
etl.spark.sql(
    '''
    select count(*)
        ,sum(case when b.retroID is null then 1 else 0 end) as mismatches
    from RETROSHEET_EVENTS as a
    left join LAHMAN_PEOPLE as b
        on a.batter=b.retroID
    '''
).show()

+--------+----------+
|count(1)|mismatches|
+--------+----------+
|  238866|         0|
+--------+----------+

